<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## 02-1 · Part 3: Continuous, Discrete, and Mixed Optimization

**The available equipment settings determine which decisions are possible before any candidate is simulated.**

Part 2 classified problems by their constraints. This part keeps the classroom simulator, objective, and requirements fixed and changes only the decision domain \(\mathcal X\).

| Kept fixed | Changed here | Resulting classification |
|:---|:---|:---|
| $y=\operatorname{Sim}(x)$, $f(y)$, $g(x,y)$ | Allowed values in $\mathcal X$ | Continuous, discrete, or mixed optimization |


### 1 · The allowed values define the decision type

The decision vector still represents early and late cooling. What changes is the set of values the controller can accept.

| Decision type | Example domain | Physical meaning |
|:---|:---|:---|
| Continuous | $x\in[0,5]^2\subset\mathbb R^2$ | Each cooling level may take any real value from 0 to 5 |
| Discrete | $x\in\{0,0.5,1,\ldots,5\}^2$ | Only listed cooling settings are available |
| Mixed | $x=(q,z)$, $q\in[0,5]$, $z\in\{0,1\}$ | A continuous level and a discrete equipment mode are chosen together |

A **mixed** problem contains at least one continuous component and at least one discrete component. If all decision components are discrete, the problem is a discrete optimization problem.

The classroom equipment is assumed to be continuously adjustable, so its physical domain is \(\mathcal X=[0,5]^2\).


### 2 · A search grid is not automatically a discrete domain

The domain describes what the real system allows. A search method describes which candidates an algorithm evaluates. These statements are different:

| Statement | Problem type | Reason |
|:---|:---|:---|
| The controller accepts every value from 0 to 5 | Continuous | The physical domain contains real values |
| The controller exposes only 0.5-unit settings | Discrete | The physical domain contains a finite set |
| The controller is continuous, but the search samples a 0.5 grid | Continuous | The grid approximates a continuous domain |

The next figure holds the state equation, performance functions, and constraints fixed. The left panel samples the continuous domain densely. The right panel treats the 0.5-unit settings as the stated discrete domain. Compare the smooth region on the left with the isolated allowable points on the right.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-1_problem_formulation/assets/03_domain_comparison.png" alt="Comparison of a densely sampled continuous cooling domain and discrete half-unit cooling settings with feasible candidates and selected grid candidates" width="780" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

Run the code below to recreate both panels and inspect the selected candidates.


In [ ]:
import numpy as np


TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0

# Fixed parameters
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45

# External inputs
OUTSIDE_TEMPERATURE = np.full(TIME_STEPS, 31.0)
OCCUPANTS = np.full(TIME_STEPS, 20.0)

# Decision domain and requirement limits
MIN_COOLING = 0.0
MAX_COOLING = 5.0
MIN_TEMPERATURE = 20.0
MAX_TEMPERATURE = 30.0
MAX_ENERGY = 60.0


def expand_decision(x):
    """Expand x=[early cooling, late cooling] into the 12-step schedule."""
    early_cooling, late_cooling = np.asarray(x, dtype=float)
    return np.r_[
        np.full(6, early_cooling),
        np.full(6, late_cooling),
    ]


def simulation_model(x):
    """Return y=Sim(x): the state path and raw performance outputs."""
    cooling_schedule = expand_decision(x)
    temperatures = [INITIAL_TEMPERATURE]

    for outdoor, people, cooling in zip(
        OUTSIDE_TEMPERATURE, OCCUPANTS, cooling_schedule
    ):
        current = temperatures[-1]
        temperatures.append(
            current
            + WEATHER_EXCHANGE * (outdoor - current)
            + OCCUPANT_HEAT * people
            - COOLING_EFFECT * cooling
        )

    temperatures = np.asarray(temperatures)
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling_schedule**2)
    return {
        "temperatures": temperatures,
        "discomfort": float(discomfort),
        "energy": float(energy),
    }


def objective_function(y, energy_weight=1.0):
    """Return f(y; lambda_E)=D(u)+lambda_E E(u)."""
    return y["discomfort"] + float(energy_weight) * y["energy"]


def inequality_constraints(x, y, energy_limit=MAX_ENERGY):
    """Return residuals in the feasible form g_j(x,y) <= 0."""
    early_cooling, late_cooling = np.asarray(x, dtype=float)
    temperatures = y["temperatures"][1:]
    return {
        "early lower": MIN_COOLING - early_cooling,
        "early upper": early_cooling - MAX_COOLING,
        "late lower": MIN_COOLING - late_cooling,
        "late upper": late_cooling - MAX_COOLING,
        "temperature lower": MIN_TEMPERATURE - temperatures.min(),
        "temperature upper": temperatures.max() - MAX_TEMPERATURE,
        "energy": y["energy"] - float(energy_limit),
    }


def equality_constraints(x, y):
    """Return state-equation residuals h_t(x,y), which should equal zero."""
    cooling_schedule = expand_decision(x)
    temperatures = y["temperatures"]
    residuals = []
    for step, (outdoor, people, cooling) in enumerate(
        zip(OUTSIDE_TEMPERATURE, OCCUPANTS, cooling_schedule)
    ):
        predicted_next = (
            temperatures[step]
            + WEATHER_EXCHANGE * (outdoor - temperatures[step])
            + OCCUPANT_HEAT * people
            - COOLING_EFFECT * cooling
        )
        residuals.append(temperatures[step + 1] - predicted_next)
    return np.asarray(residuals)


def evaluate_formulation(x, energy_weight=1.0, energy_limit=MAX_ENERGY):
    """Evaluate x, y, f, g, and h for one candidate decision."""
    x = tuple(map(float, x))
    y = simulation_model(x)
    g = inequality_constraints(x, y, energy_limit)
    h = equality_constraints(x, y)
    feasible = all(value <= 1e-10 for value in g.values()) and np.allclose(h, 0.0)
    return {
        "x": x,
        "y": y,
        "f": objective_function(y, energy_weight),
        "g": g,
        "h": h,
        "energy_weight": float(energy_weight),
        "energy_limit": float(energy_limit),
        "feasible": bool(feasible),
    }


def enumerate_candidates(step=0.5, energy_weight=1.0, energy_limit=MAX_ENERGY):
    """Evaluate a stated finite candidate grid."""
    levels = np.arange(MIN_COOLING, MAX_COOLING + step / 2, step)
    return [
        evaluate_formulation((early, late), energy_weight, energy_limit)
        for early in levels
        for late in levels
    ]


def select_best(records, key="f"):
    """Select the lowest-valued feasible record for the requested key."""
    return min(
        (record for record in records if record["feasible"]),
        key=lambda record: record[key] if key != "discomfort" else record["y"][key],
    )


def pareto_front(records):
    """Return nondominated feasible records for minimizing discomfort and energy."""
    feasible_records = [record for record in records if record["feasible"]]
    nondominated = []
    for candidate in feasible_records:
        candidate_d = candidate["y"]["discomfort"]
        candidate_e = candidate["y"]["energy"]
        dominated = any(
            other["y"]["discomfort"] <= candidate_d
            and other["y"]["energy"] <= candidate_e
            and (
                other["y"]["discomfort"] < candidate_d
                or other["y"]["energy"] < candidate_e
            )
            for other in feasible_records
        )
        if not dominated:
            nondominated.append(candidate)
    return sorted(nondominated, key=lambda record: record["y"]["energy"])

import sys
from types import SimpleNamespace

import matplotlib

def _pyplot(*, interactive=False):
    """Return pyplot, activating ipympl for interactive figures when available."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry

            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt

    return plt

def show_domain_comparison(
    enumerate_candidates,
    select_best,
    *,
    min_cooling=0.0,
    max_cooling=5.0,
    max_energy=60.0,
):
    """Compare a densely sampled continuous domain with a discrete domain."""
    plt = _pyplot()

    def landscape(step):
        records = enumerate_candidates(step, 1.0, max_energy)
        levels = np.arange(min_cooling, max_cooling + step / 2, step)
        values = np.full((len(levels), len(levels)), np.nan)
        for record in records:
            early, late = record["x"]
            if record["feasible"]:
                values[
                    int(round((early - min_cooling) / step)),
                    int(round((late - min_cooling) / step)),
                ] = record["f"]
        return levels, values, records

    dense_levels, dense_values, dense_records = landscape(0.05)
    grid_levels, grid_values, grid_records = landscape(0.5)
    dense_best = select_best(dense_records)
    grid_best = select_best(grid_records)

    figure, axes = plt.subplots(1, 2, figsize=(10.6, 4.6), layout="constrained")
    color_map = plt.colormaps["viridis_r"].copy()
    color_map.set_bad("#e6e6e6")
    image = axes[0].imshow(
        dense_values,
        origin="lower",
        cmap=color_map,
        extent=(min_cooling - 0.025, max_cooling + 0.025) * 2,
        aspect="equal",
    )
    axes[0].scatter(
        dense_best["x"][1], dense_best["x"][0],
        marker="*", s=190, color="gold", edgecolor="black",
        label="Best 0.05-grid sample",
    )
    axes[0].set_title("Continuous domain · densely sampled view")
    axes[0].legend(fontsize=8)

    feasible = [record for record in grid_records if record["feasible"]]
    infeasible = [record for record in grid_records if not record["feasible"]]
    axes[1].scatter(
        [record["x"][1] for record in infeasible],
        [record["x"][0] for record in infeasible],
        marker="x", color="lightgray", label="Infeasible",
    )
    axes[1].scatter(
        [record["x"][1] for record in feasible],
        [record["x"][0] for record in feasible],
        c=[record["f"] for record in feasible],
        cmap=color_map,
        edgecolor="white",
        s=65,
        label="Feasible",
    )
    axes[1].scatter(
        grid_best["x"][1], grid_best["x"][0],
        marker="*", s=190, color="gold", edgecolor="black",
        label="Best discrete candidate",
    )
    axes[1].set_title("Discrete domain · 0.5-unit settings")
    axes[1].legend(fontsize=8)
    for axis in axes:
        axis.set(
            xlabel="Late cooling $x_2$",
            ylabel="Early cooling $x_1$",
            xlim=(min_cooling - 0.2, max_cooling + 0.2),
            ylim=(min_cooling - 0.2, max_cooling + 0.2),
            xticks=np.arange(min_cooling, max_cooling + 0.1, 1),
            yticks=np.arange(min_cooling, max_cooling + 0.1, 1),
        )
        axis.grid(alpha=0.18)
    figure.colorbar(
        image,
        ax=axes.ravel().tolist(),
        label="Objective value $f$",
        shrink=0.84,
        pad=0.03,
    )
    figure.suptitle(
        "The domain is part of the formulation; the grid is part of the search",
        fontsize=12,
    )
    plt.show()
    plt.close(figure)
    print(
        f"Best 0.05-grid sample: x={dense_best['x']}, f={dense_best['f']:.2f}\n"
        f"Best 0.5-grid candidate: x={grid_best['x']}, f={grid_best['f']:.2f}"
    )
    return SimpleNamespace(
        dense_best=dense_best,
        grid_best=grid_best,
        dense_records=dense_records,
        grid_records=grid_records,
    )

domain_comparison = show_domain_comparison(
    enumerate_candidates,
    select_best,
    min_cooling=MIN_COOLING,
    max_cooling=MAX_COOLING,
    max_energy=MAX_ENERGY,
)


### 3 · Changing the domain changes the candidates that can be selected

A continuous domain contains every point inside the square \([0,5]^2\). A discrete domain contains only the displayed settings. The best discrete candidate must be one of those settings, while a continuous solution may lie between them.

The dense sample in the left panel is still finite. Its highlighted point is the best **sampled candidate**, not a guaranteed continuous optimum. Establishing the continuous optimum requires an appropriate continuous optimization method or a mathematical proof.

Changing \(\mathcal X\) does not change the physical response of a fixed decision. If \(x=[3,2]^{\mathsf{T}}\) belongs to both domains, it produces the same \(T\), \(D\), and \(E\) in both formulations. Only the set of competing candidates changes.


### 4 · Mixed decisions combine amounts and choices

Suppose the controller chooses a continuous cooling level \(q\in[0,5]\) and a discrete operating mode \(z\in\{0,1\}\). Then

> $\displaystyle x=(q,z)\in[0,5]\times\{0,1\}.$

The variable \(q\) answers “how much?” while \(z\) answers “which mode?” This is a mixed optimization problem even if both components enter the same objective and constraints.

A grid used to approximate \(q\) does not change its physical meaning. The problem becomes fully discrete only if the equipment also restricts \(q\) to listed settings.


### Takeaway

Classify the decision domain by asking what values the real decision may take:

> **real-valued components only → continuous · listed choices only → discrete · both types → mixed**

Keep the domain separate from the search method. A finite grid may approximate a continuous problem, and its best point is only the best grid candidate.

Part 4 keeps the decision domain fixed and changes the objective structure.
